# 35 — AI Governance, Compliance & Risk Frameworks — Lab

## Description

This lab conducts a structured algorithmic audit on a hiring-decision classifier and produces a complete AI Impact Assessment for a high-risk deployment scenario. You will compute accuracy and fairness metrics, review model transparency evidence, fill an audit report template, draft an AIIA, and formulate a deployment recommendation based on audit findings.

## Objectives

- Conduct a structured algorithmic audit of a classifier covering accuracy, fairness, transparency, and accountability dimensions.
- Produce a complete AI Impact Assessment document for a high-risk AI deployment scenario, including system description, risk identification, affected populations, mitigation plan, and residual risk statement.

## Prerequisites

You should have completed the lesson 35 notes and be familiar with the NIST AI RMF, EU AI Act risk classification, the four audit dimensions (accuracy, fairness, transparency, accountability), and the five AI Impact Assessment components. You should also have basic familiarity with fairness metrics from lesson 32 (Demographic Parity, Equalized Odds) and explainability outputs from lesson 34 (SHAP).

## Required Software/Packages

- Python 3.8+
- pandas
- numpy
- scikit-learn
- matplotlib

## Environment/Setup Instructions

All required libraries are pre-installed in the standard environment. No additional setup is required.

## Background/Context

Imagine you are a governance specialist at a mid-sized technology company evaluating a proposed hiring classifier. The model will screen resumes to decide which candidates advance to phone interviews. Your task is to audit the model against four audit dimensions, produce an AI Impact Assessment with five required components, and make a deployment recommendation. The model was trained on 10 years of historical hiring decisions from your company. Approximately 30 percent of the training data comes from applications to engineering roles, 25 percent from sales, 20 percent from product, and 25 percent from other roles.

The model achieves roughly 81 percent overall accuracy on a held-out test set. However, preliminary fairness analysis suggests disparities exist across demographic groups. Your audit will quantify these disparities, assess mitigation strategies, and determine whether the model meets compliance standards before deployment.

## Exercise Instructions

1. Generate the hiring dataset and train the classifier; compute overall accuracy and accuracy disaggregated by gender and race, and visualize the disparities. 
2. Compute the Demographic Parity difference and Equalized Odds difference step by step with pandas/NumPy; load and examine a pre-computed SHAP feature-importance summary. 
3. Using the audit template provided below, fill in findings for all four dimensions (accuracy, fairness, transparency, accountability); assign a compliance status (pass / flag / fail) per dimension. 
4. Draft the AI Impact Assessment with all five sections: system description, risk identification, affected populations, mitigation plan, and residual risk statement; conclude with a deployment recommendation (deploy / conditional deploy / do not deploy). 

## Setup: Libraries and Synthetic Hiring Dataset

In [ ]:
# Setup: import libraries and fix the random seed for reproducibility

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

np.random.seed(42)

### Generate the synthetic hiring dataset

We simulate 1,000 candidates. The historical hiring rule is deliberately biased: a qualified candidate (GPA, interview score, experience) is hired only if they are male **or** White, and 15% label noise is added. The classifier will learn this bias from the data.

In [ ]:
# Synthetic hiring dataset (1,000 candidates)
# Features: gpa, years_experience, interview_score,
#           gender (0 = Female, 1 = Male), race (0 = Non-White, 1 = White)
# Label:    hired (0 = No, 1 = Yes)

n_samples = 1000

df = pd.DataFrame({
    'gpa': np.random.uniform(2.5, 4.0, n_samples),
    'years_experience': np.random.randint(0, 20, n_samples),
    'interview_score': np.random.uniform(1, 10, n_samples),
    'gender': np.random.choice([0, 1], n_samples, p=[0.4, 0.6]),  # 40% Female, 60% Male
    'race': np.random.choice([0, 1], n_samples, p=[0.3, 0.7]),    # 30% Non-White, 70% White
})

# Historical hiring decisions with built-in bias:
# a qualified candidate is hired only if they are male OR White
df['hired'] = (
    (df['gpa'] > 3.2) &
    (df['interview_score'] > 5.5) &
    (df['years_experience'] > 2) &
    ((df['gender'] == 1) | (df['race'] == 1))
).astype(int)

# Add label noise: flip 15% of the hiring decisions at random
noise_idx = np.random.choice(len(df), size=int(0.15 * len(df)), replace=False)
df.loc[noise_idx, 'hired'] = 1 - df.loc[noise_idx, 'hired']

print(f"Dataset shape: {df.shape}")
print(f"Overall hire rate: {df['hired'].mean():.2%}")
df.head()

### Visualize the historical bias

Before training anything, plot hire rates by group. The gaps below are in the *labels* the model will learn from.

In [ ]:
# Visualize historical hire rates by protected group
fig, axes = plt.subplots(1, 2, figsize=(10, 3.5), sharey=True)

for ax, attr, names in [(axes[0], 'gender', ['Female', 'Male']),
                        (axes[1], 'race', ['Non-White', 'White'])]:
    rates = [df.loc[df[attr] == v, 'hired'].mean() for v in [0, 1]]
    bars = ax.bar(names, rates, color=['#d62728', '#1f77b4'])
    ax.bar_label(bars, fmt='%.3f')
    ax.set_title(f'Hire rate by {attr}')
    ax.set_ylabel('Hire rate')

plt.suptitle('Historical bias is already visible in the training data')
plt.tight_layout()
plt.show()

### Train the hiring classifier

Split 70/30, standardize the features, and fit a logistic regression. Note that the protected attributes (`gender`, `race`) are included as model inputs — a deliberate design flaw that this audit should catch.

In [ ]:
# Split the data, scale the features, and train a logistic regression classifier
features = ['gpa', 'years_experience', 'interview_score', 'gender', 'race']

train_df, test_df = train_test_split(df, test_size=0.3, random_state=42)
X_train, y_train = train_df[features], train_df['hired']
X_test, y_test = test_df[features], test_df['hired']

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

model = LogisticRegression(random_state=42, max_iter=1000)
model.fit(X_train_scaled, y_train)

y_pred_test = model.predict(X_test_scaled)

# Keep test data and predictions together for the audit steps
test_results = test_df.copy()
test_results['predicted_hired'] = y_pred_test

print(f"Test set size: {len(test_df)}")
print(f"Overall test accuracy: {(y_pred_test == y_test).mean():.2%}")

## Step 1: Compute Accuracy Disaggregated by Protected Group

Now compute accuracy overall and separately for each gender and race group. Report exact percentages.

In [ ]:
# Step 1: compute accuracy overall and disaggregated by gender and race

overall_accuracy = (y_pred_test == y_test).mean()
print(f"Overall accuracy: {overall_accuracy:.4f}")

accuracy_by_gender = {}
print("\nAccuracy by gender:")
for g, name in [(0, 'Female'), (1, 'Male')]:
    mask = test_df['gender'] == g
    accuracy_by_gender[g] = (y_pred_test[mask] == y_test[mask]).mean()
    print(f"  {name}: {accuracy_by_gender[g]:.4f} (n={mask.sum()})")

accuracy_by_race = {}
print("\nAccuracy by race:")
for r, name in [(0, 'Non-White'), (1, 'White')]:
    mask = test_df['race'] == r
    accuracy_by_race[r] = (y_pred_test[mask] == y_test[mask]).mean()
    print(f"  {name}: {accuracy_by_race[r]:.4f} (n={mask.sum()})")

gender_disparity = abs(accuracy_by_gender[0] - accuracy_by_gender[1])
race_disparity = abs(accuracy_by_race[0] - accuracy_by_race[1])

print(f"\nAccuracy disparity by gender: {gender_disparity:.4f}")
print(f"Accuracy disparity by race: {race_disparity:.4f}")

### Visualize disaggregated accuracy

Compare per-group accuracy against the overall accuracy line.

In [ ]:
# Visualize disaggregated accuracy against the overall accuracy
groups = ['Female', 'Male', 'Non-White', 'White']
values = [accuracy_by_gender[0], accuracy_by_gender[1],
          accuracy_by_race[0], accuracy_by_race[1]]

fig, ax = plt.subplots(figsize=(7, 3.5))
bars = ax.bar(groups, values, color=['#d62728', '#1f77b4', '#d62728', '#1f77b4'])
ax.bar_label(bars, fmt='%.3f')
ax.axhline(overall_accuracy, color='black', linestyle='--',
           label=f'Overall accuracy ({overall_accuracy:.3f})')
ax.set_ylim(0.5, 1.0)
ax.set_ylabel('Accuracy')
ax.set_title('Accuracy disaggregated by protected group')
ax.legend()
plt.tight_layout()
plt.show()

## Step 2: Compute Fairness Metrics and Review SHAP Evidence

Compute the Demographic Parity difference and the Equalized Odds difference step by step. Then examine a pre-computed SHAP visualization as transparency evidence.

$$\Delta_{\mathrm{DP}} = \left| \, P(\hat{Y}=1 \mid \text{Female}) - P(\hat{Y}=1 \mid \text{Male}) \, \right|
\qquad
\Delta_{\mathrm{EO}} = \max\left( \left| \mathrm{TPR}_F - \mathrm{TPR}_M \right|, \; \left| \mathrm{FPR}_F - \mathrm{FPR}_M \right| \right)$$

In [ ]:
# Step 2a: Demographic Parity — compare positive prediction rates across gender groups

FAIRNESS_THRESHOLD = 0.05

pos_rate = {}
print("Demographic Parity analysis (by gender):")
for g, name in [(0, 'Female'), (1, 'Male')]:
    mask = test_results['gender'] == g
    pos_rate[g] = (test_results.loc[mask, 'predicted_hired'] == 1).mean()
    print(f"  {name}: positive prediction rate = {pos_rate[g]:.4f}")

dem_parity_gap = abs(pos_rate[0] - pos_rate[1])
print(f"\nDemographic Parity difference: {dem_parity_gap:.4f}")
print(f"Threshold: {FAIRNESS_THRESHOLD}")
print(f"Status: {'PASS' if dem_parity_gap <= FAIRNESS_THRESHOLD else 'FLAG'}")

In [ ]:
# Step 2b: Equalized Odds — compare TPR and FPR across gender groups

def tpr_fpr(actual, predicted):
    """Return (TPR, FPR) for one group."""
    tp = ((predicted == 1) & (actual == 1)).sum()
    fn = ((predicted == 0) & (actual == 1)).sum()
    fp = ((predicted == 1) & (actual == 0)).sum()
    tn = ((predicted == 0) & (actual == 0)).sum()
    tpr = tp / (tp + fn) if (tp + fn) > 0 else 0.0
    fpr = fp / (fp + tn) if (fp + tn) > 0 else 0.0
    return tpr, fpr

rates = {}
print("Equalized Odds analysis (by gender):")
for g, name in [(0, 'Female'), (1, 'Male')]:
    mask = test_results['gender'] == g
    tpr, fpr = tpr_fpr(y_test[mask], test_results.loc[mask, 'predicted_hired'])
    rates[g] = {'tpr': tpr, 'fpr': fpr}
    print(f"  {name}: TPR = {tpr:.4f}, FPR = {fpr:.4f}")

tpr_gap = abs(rates[0]['tpr'] - rates[1]['tpr'])
fpr_gap = abs(rates[0]['fpr'] - rates[1]['fpr'])
equalized_odds_gap = max(tpr_gap, fpr_gap)

print(f"\nTPR difference: {tpr_gap:.4f}")
print(f"FPR difference: {fpr_gap:.4f}")
print(f"Equalized Odds difference (max): {equalized_odds_gap:.4f}")
print(f"Threshold: {FAIRNESS_THRESHOLD}")
print(f"Status: {'PASS' if equalized_odds_gap <= FAIRNESS_THRESHOLD else 'FLAG'}")

### Visualize the fairness gaps

In [ ]:
# Visualize the fairness gaps between the two gender groups
metrics = ['Positive rate', 'TPR', 'FPR']
female_vals = [pos_rate[0], rates[0]['tpr'], rates[0]['fpr']]
male_vals = [pos_rate[1], rates[1]['tpr'], rates[1]['fpr']]

x = np.arange(len(metrics))
width = 0.35

fig, ax = plt.subplots(figsize=(7, 3.5))
b1 = ax.bar(x - width / 2, female_vals, width, label='Female', color='#d62728')
b2 = ax.bar(x + width / 2, male_vals, width, label='Male', color='#1f77b4')
ax.bar_label(b1, fmt='%.3f')
ax.bar_label(b2, fmt='%.3f')
ax.set_xticks(x)
ax.set_xticklabels(metrics)
ax.set_title(f'Fairness metrics by gender (gaps above {FAIRNESS_THRESHOLD} are flagged)')
ax.legend()
plt.tight_layout()
plt.show()

## Transparency Evidence: SHAP Analysis

Below is a pre-computed summary of SHAP feature importance from the hiring classifier. Feature importance shows which model inputs most influence hiring predictions.

In [ ]:
# Pre-computed SHAP summary (from external analysis)
# This simulates SHAP output from explainability analysis

shap_importance = {
    'interview_score': 0.35,
    'gpa': 0.28,
    'years_experience': 0.18,
    'gender': 0.12,  # Concerning: gender should not be a top feature
    'race': 0.07
}

print("SHAP Feature Importance (Transparency Evidence):")
print("="*50)
for feature, importance in sorted(shap_importance.items(), key=lambda x: x[1], reverse=True):
    print(f"  {feature}: {importance:.2%}")

print(f"\nAnalysis:")
print(f"  Top 3 features account for {sum(list(shap_importance.values())[:3]):.2%} of model variance")
print(f"  Gender feature importance: {shap_importance['gender']:.2%} (concerning)")
print(f"  Race feature importance: {shap_importance['race']:.2%} (acceptable)")
print(f"\nFinding: Gender appears as a top-4 feature, suggesting the model may rely on demographic")
print(f"proxies. This requires investigation of potential indirect discrimination.")

### Visualize SHAP feature importance

In [ ]:
# Visualize SHAP importance — demographic features highlighted in red
sorted_items = sorted(shap_importance.items(), key=lambda x: x[1])
names = [f for f, _ in sorted_items]
vals = [v for _, v in sorted_items]
colors = ['#d62728' if f in ('gender', 'race') else '#1f77b4' for f in names]

fig, ax = plt.subplots(figsize=(7, 3))
bars = ax.barh(names, vals, color=colors)
ax.bar_label(bars, fmt='%.2f')
ax.set_xlabel('SHAP feature importance (share of model variance)')
ax.set_title('Transparency evidence: demographic features (red) should not rank high')
plt.tight_layout()
plt.show()

## Step 3: Audit Template — Fill in Your Findings

Use the audit template below to document findings across the four dimensions. Compute exact values and assign compliance status (pass / flag / fail).

In [ ]:
# Step 3a: document findings for the accuracy and fairness dimensions

accuracy_findings = {
    "findings": f"""Overall accuracy: {overall_accuracy:.2%}
Accuracy by gender — Female: {accuracy_by_gender[0]:.2%}, Male: {accuracy_by_gender[1]:.2%} (disparity: {gender_disparity:.2%})
Accuracy by race — Non-White: {accuracy_by_race[0]:.2%}, White: {accuracy_by_race[1]:.2%} (disparity: {race_disparity:.2%})
Threshold: 5.00%
Assessment: group accuracy disparities are below the 5-percentage-point threshold,
but error rates still differ across groups and should be monitored.""",
    "status": "flag" if max(gender_disparity, race_disparity) > 0.05 else "pass",
}

fairness_findings = {
    "findings": f"""Demographic Parity difference (gender): {dem_parity_gap:.4f}
Equalized Odds TPR difference: {tpr_gap:.4f}
Equalized Odds FPR difference: {fpr_gap:.4f}
Equalized Odds difference (max): {equalized_odds_gap:.4f}
Threshold: 0.05
Assessment: Both Demographic Parity and Equalized Odds show gaps above acceptable thresholds.
The model systematically produces different outcomes for different demographic groups.""",
    "status": "flag" if (dem_parity_gap > 0.05 or equalized_odds_gap > 0.05) else "pass",
}

print("Accuracy dimension status:", accuracy_findings["status"])
print("Fairness dimension status:", fairness_findings["status"])

### Step 3b: Transparency and accountability findings

These two dimensions use the SHAP evidence from Step 2 and the organizational safeguards described in the background scenario.

In [ ]:
# Step 3b: document findings for the transparency and accountability dimensions

transparency_findings = {
    "findings": f"""SHAP top-3 features: interview_score (35%), gpa (28%), years_experience (18%)
Feature examination: interview_score, gpa, and years_experience are job-related and appropriate.
Demographic features: gender ({shap_importance['gender']:.0%}), race ({shap_importance['race']:.0%}) — both present in top features.
Assessment: Gender ranks as the 4th most important feature.
This suggests potential use of demographic information (directly or via proxies) in predictions.
Flag for investigation: model may be using demographic proxies against policy.""",
    "status": "flag",
}

accountability_findings = {
    "findings": """Human review process: weekly manual review of top 100 candidates by senior hiring manager documented.
Appeal procedure: candidates rejected can submit appeals; reviewed by HR director.
Audit trail: all model decisions logged with timestamp and features used.
Authority: Chief AI Officer approval required before deployment (documented in governance policy).
Escalation: model performance monitored quarterly; retraining triggered if gender disparity exceeds 8%.
Assessment: organizational safeguards are partially in place but need strengthening.
No independent third-party audit has been conducted yet.""",
    "status": "flag",
}

print("Transparency dimension status:", transparency_findings["status"])
print("Accountability dimension status:", accountability_findings["status"])

### Step 3c: Assemble and print the audit report

In [ ]:
# Step 3c: assemble the audit report and print it

audit_report = {
    "lesson_id": "lesson_35",
    "audit_date": "2026-08-20",
    "system_name": "Hiring Decision Classifier",
    "findings": {
        "accuracy_dimension": accuracy_findings,
        "fairness_dimension": fairness_findings,
        "transparency_dimension": transparency_findings,
        "accountability_dimension": accountability_findings,
    },
    "overall_recommendation": "conditional_deploy",
    "reasoning": """The classifier shows concerning fairness disparities across demographic groups (gender and race).
Demographic Parity and Equalized Odds thresholds are exceeded. Demographic features appear to influence
predictions. However, human review safeguards and audit procedures are documented. Recommend conditional
deployment pending: (1) retrain model with fairness constraints, (2) increase manual review scope to top 500
candidates, (3) conduct third-party algorithmic audit, (4) establish performance monitoring dashboard,
(5) implement bias remediation procedures in HR workflows. Revisit deployment decision in 30 days.""",
}

print("ALGORITHMIC AUDIT REPORT")
print("=" * 70)
print(f"System: {audit_report['system_name']}")
print(f"Audit date: {audit_report['audit_date']}")

for dimension, data in audit_report['findings'].items():
    print(f"\n{dimension.upper().replace('_', ' ')}")
    print(f"Status: [{data['status'].upper()}]")
    print(data['findings'])

print(f"\nOVERALL RECOMMENDATION: {audit_report['overall_recommendation'].upper().replace('_', ' ')}")
print("\nREASONING:")
print(audit_report['reasoning'])

# Keep a reference for the verification cell at the end of the lab
audit_findings = audit_report

### Visual summary of the audit

In [ ]:
# Visual summary: compliance status per audit dimension
status_colors = {'pass': '#2ca02c', 'flag': '#ff7f0e', 'fail': '#d62728'}
labels = [d.replace('_dimension', '').capitalize() for d in audit_report['findings']]
statuses = [data['status'] for data in audit_report['findings'].values()]

fig, ax = plt.subplots(figsize=(7, 2.2))
ax.barh(labels, [1] * len(labels), color=[status_colors[s] for s in statuses])
for i, s in enumerate(statuses):
    ax.text(0.5, i, s.upper(), ha='center', va='center', color='white', fontweight='bold')
ax.set_xticks([])
ax.set_xlim(0, 1)
ax.invert_yaxis()
ax.set_title(f"Audit summary — overall: "
             f"{audit_report['overall_recommendation'].replace('_', ' ').upper()}")
plt.tight_layout()
plt.show()

## Step 4: AI Impact Assessment

Now draft the complete AI Impact Assessment for this hiring system. Include all five required components: system description, risk identification, affected populations, mitigation plan, and residual risk statement. Conclude with a deployment recommendation justified by audit findings.

In [ ]:
# Step 4a: AIIA — system description and risk identification

system_description = """The Hiring Decision Classifier is a logistic regression model trained on 10 years of historical hiring decisions
across 500+ job listings (30% engineering, 25% sales, 20% product, 25% other roles). The model predicts whether a candidate
should advance to phone interview based on resume data (GPA, years of experience, interview score) and automatically categorizes
candidates into interview/no-interview decisions. The system is deployed as part of the resume-screening workflow used by the
recruiting team to process 10,000+ applications per quarter."""

risk_identification = [
    {
        "risk": "Demographic bias in predictions",
        "affected_dimension": "fairness",
        "likelihood": "high",
        "severity": "high",
        "evidence": f"Gender accuracy disparity: {gender_disparity:.2%}. Race accuracy disparity: {race_disparity:.2%}. "
                    f"Demographic Parity difference: {dem_parity_gap:.4f}.",
        "residual_risk_after_mitigation": "medium",
    },
    {
        "risk": "Model uses demographic proxies",
        "affected_dimension": "transparency",
        "likelihood": "high",
        "severity": "high",
        "evidence": f"Gender feature ranks 4th in SHAP importance ({shap_importance['gender']:.1%}). "
                    "Training data reflects historical bias in hiring.",
        "residual_risk_after_mitigation": "medium",
    },
    {
        "risk": "Disparate impact on minority candidates",
        "affected_dimension": "accountability",
        "likelihood": "high",
        "severity": "high",
        "evidence": "Positive prediction rate for Non-White candidates is roughly half that of White candidates "
                    "(about 8% vs. 17% on the test set), reproducing the historical hiring gap.",
        "residual_risk_after_mitigation": "medium",
    },
    {
        "risk": "Data quality issues",
        "affected_dimension": "accuracy",
        "likelihood": "medium",
        "severity": "medium",
        "evidence": "Training data drawn from historical hiring decisions, which themselves contain bias "
                    "and may not represent optimal hiring practices.",
        "residual_risk_after_mitigation": "medium",
    },
]

print(f"System description drafted ({len(system_description)} characters).")
print(f"Risks identified: {len(risk_identification)}")

### Step 4b: Affected populations and mitigation plan

In [ ]:
# Step 4b: AIIA — affected populations and mitigation plan

affected_populations = """The model directly affects hiring outcomes for all candidates. Disproportionately affected groups include:
Female candidates: receive positive predictions at well under half the rate of male candidates and have a much lower true positive rate.
Non-White candidates: receive positive predictions at roughly half the rate of White candidates.
Candidates from economically disadvantaged backgrounds: may have lower GPA and years of experience due to structural barriers; model amplifies this disadvantage.
These populations face reduced interview opportunities, limiting employment and economic mobility."""

mitigation_plan = """Mitigation 1: Model Retraining with Fairness Constraints
Retrain the logistic regression model with demographic parity constraints. Target threshold: gender/race demographic parity difference <= 0.05.
Timeline: 2 weeks. Responsibility: ML Engineering team.

Mitigation 2: Human-in-the-Loop Review
Increase manual human review scope from top 100 to top 500 candidates. Senior hiring managers review flagged predictions.
All interview/no-interview decisions pass through human review before HR sees them.
Timeline: Immediate. Responsibility: Recruiting team (Chief Recruiter).

Mitigation 3: Third-Party Algorithmic Audit
Engage independent external auditor to conduct full algorithmic audit covering accuracy, fairness, transparency, and accountability.
Auditor scope: review model card, conduct statistical testing, verify audit trail completeness.
Timeline: 4 weeks. Responsibility: Chief AI Officer.

Mitigation 4: Monitoring and Escalation
Establish quarterly performance monitoring dashboard tracking gender/race accuracy and fairness metrics across all demographic groups.
Escalation trigger: if demographic parity difference exceeds 8% or accuracy disparity exceeds 7%, halt model deployment and retrain.
Timeline: Ongoing. Responsibility: Data Science team.

Mitigation 5: Bias Remediation Procedures
Establish HR escalation procedure: candidates rejected by model can submit bias appeals. Appeals reviewed within 5 business days.
Candidates may provide additional information (new certifications, projects) for reconsideration.
Timeline: Immediate. Responsibility: Chief HR Officer."""

print("Affected populations and mitigation plan drafted.")

### Step 4c: Residual risk statement and deployment recommendation

In [ ]:
# Step 4c: AIIA — residual risk statement, deployment recommendation, and assembly

residual_risk_statement = """After implementing mitigations 1-5 above, residual risks remain:

Residual Risk 1: Demographic Parity Gap. After fairness-constrained retraining, target demographic parity difference is <= 0.05.
However, a residual gap of 3-5% is expected due to inherent data biases. Risk Level: Medium.

Residual Risk 2: Model still influences recommendation order. Even with human review, the model determines which candidates appear
first in the review list. Hiring managers' unconscious biases may amplify model predictions (anchoring effect).
Risk Level: Medium.

Residual Risk 3: Historical bias embedded in training data. The 10-year training dataset reflects past discriminatory practices.
Even with fairness constraints, the model cannot fully eliminate this historical bias.
Risk Level: Medium.

Residual Risk Mitigation: Quarterly audits and appeals reviews are required to track residual risk over time.
If residual risks exceed stated thresholds after 2 quarters, system will be paused for model redesign or replacement.

EU AI Act Classification: High-Risk (Employment category, Annex III). Mandatory compliance requirements apply."""

aiia = {
    "title": "AI Impact Assessment: Hiring Decision Classifier",
    "system_description": system_description,
    "risk_identification": risk_identification,
    "affected_populations": affected_populations,
    "mitigation_plan": mitigation_plan,
    "residual_risk_statement": residual_risk_statement,
    "deployment_recommendation": "CONDITIONAL DEPLOY",
}

print("AIIA assembled with all five components plus a deployment recommendation.")

### Visualize the risk matrix

Each identified risk is placed on a likelihood × severity grid (green = low concern, red = high concern).

In [ ]:
# Risk matrix: likelihood vs. severity for the identified risks
level = {'low': 1, 'medium': 2, 'high': 3}

fig, ax = plt.subplots(figsize=(6, 4.5))
for cx in (1, 2, 3):
    for cy in (1, 2, 3):
        score = cx * cy
        color = '#2ca02c' if score <= 2 else ('#ff7f0e' if score <= 4 else '#d62728')
        ax.add_patch(plt.Rectangle((cx - 0.5, cy - 0.5), 1, 1, color=color, alpha=0.3))

offsets = [(10, 8), (10, -16), (-28, 8), (10, 8)]
for k, risk in enumerate(aiia['risk_identification'], 1):
    rx, ry = level[risk['likelihood']], level[risk['severity']]
    ax.scatter(rx, ry, s=100, color='black', zorder=3)
    ax.annotate(f"R{k}", (rx, ry), textcoords='offset points',
                xytext=offsets[(k - 1) % len(offsets)], fontweight='bold')

ax.set_xticks([1, 2, 3])
ax.set_xticklabels(['low', 'medium', 'high'])
ax.set_yticks([1, 2, 3])
ax.set_yticklabels(['low', 'medium', 'high'])
ax.set_xlim(0.5, 3.5)
ax.set_ylim(0.5, 3.5)
ax.set_xlabel('Likelihood')
ax.set_ylabel('Severity')
ax.set_title('AIIA risk matrix')
plt.tight_layout()
plt.show()

for k, risk in enumerate(aiia['risk_identification'], 1):
    print(f"R{k}: {risk['risk']} (likelihood={risk['likelihood']}, severity={risk['severity']})")

### Print the full AIIA document

In [ ]:
# Print the full AI Impact Assessment document

print("AI IMPACT ASSESSMENT")
print("=" * 70)
print(f"Title: {aiia['title']}")

print("\nSYSTEM DESCRIPTION")
print("-" * 70)
print(aiia['system_description'])

print("\nRISK IDENTIFICATION")
print("-" * 70)
for i, risk in enumerate(aiia['risk_identification'], 1):
    print(f"\nRisk {i}: {risk['risk']}")
    print(f"  Dimension: {risk['affected_dimension']}")
    print(f"  Likelihood: {risk['likelihood']} | Severity: {risk['severity']}")
    print(f"  Evidence: {risk['evidence']}")
    print(f"  Residual risk: {risk['residual_risk_after_mitigation']}")

print("\nAFFECTED POPULATIONS")
print("-" * 70)
print(aiia['affected_populations'])

print("\nMITIGATION PLAN")
print("-" * 70)
print(aiia['mitigation_plan'])

print("\nRESIDUAL RISK STATEMENT")
print("-" * 70)
print(aiia['residual_risk_statement'])

print(f"\nDEPLOYMENT RECOMMENDATION: {aiia['deployment_recommendation']}")
print("""
Justification: The classifier demonstrates fairness concerns and demographic disparities. However, organizational
safeguards (human review, monitoring, escalation) are documented and feasible to implement within 2-4 weeks.
Conditional deployment with mandatory mitigations enables controlled rollout with oversight. Full deployment is
recommended only after a third-party audit confirms fairness constraints are effective and residual disparities
remain within acceptable thresholds.""")

## Expected Outputs

- **Step 1 (Accuracy)**: Overall accuracy approximately $81\%$. Accuracy disparities by gender and race of roughly $3$–$4$ percentage points — below the $5$-point threshold, so the accuracy dimension passes.
- **Step 2 (Fairness & Transparency)**: Demographic Parity difference $\approx 0.11$ and Equalized Odds difference $\approx 0.17$, both far above the $0.05$ threshold. SHAP shows gender in the top-4 features, confirming the transparency concern.
- **Step 3 (Audit Report)**: Four dimensions assessed with a status each — accuracy passes; fairness, transparency, and accountability are flagged. Overall recommendation is conditional_deploy.
- **Step 4 (AIIA)**: Five-section AIIA complete with explicit risk statement, affected populations, and mitigations. Deployment recommendation clearly stated.

## Comprehension Questions

1. Based on the audit findings, why does the model's demographic parity difference matter for hiring compliance? What does a difference of roughly $0.11$ mean in practical terms?
2. In the AIIA, why are "affected populations" documented separately from "risk identification"? What role does naming affected groups play in governance?

In [ ]:
# Verification: Algorithmic Audit Completeness Check
# Verify that audit report contains all four dimensions and at least one status per dimension

required_dimensions = {'accuracy_dimension', 'fairness_dimension', 'transparency_dimension', 'accountability_dimension'}
required_fields = {'findings', 'status'}

print("Verification: Audit Report Completeness")
print("="*50)

audit_complete = True
for dimension in required_dimensions:
    if dimension in audit_findings['findings']:
        data = audit_findings['findings'][dimension]
        has_fields = all(field in data for field in required_fields)
        has_valid_status = data['status'] in ['pass', 'flag', 'fail']
        print(f"  {dimension}: OK (status={data['status']})")
        if not (has_fields and has_valid_status):
            audit_complete = False
    else:
        print(f"  {dimension}: MISSING")
        audit_complete = False

if audit_complete:
    print(f"\nRESULT: PASS — Audit report contains all four dimensions with valid status.")
else:
    print(f"\nRESULT: FAIL — Audit report is incomplete.")

In [ ]:
# Verification: AIIA Completeness Check
# Verify that AIIA contains all five required components

required_components = {
    'system_description': str,
    'risk_identification': list,
    'affected_populations': str,
    'mitigation_plan': str,
    'residual_risk_statement': str
}

print("Verification: AI Impact Assessment Completeness")
print("="*50)

aiia_complete = True
for component, expected_type in required_components.items():
    if component in aiia:
        value = aiia[component]
        is_valid_type = isinstance(value, expected_type)
        is_nonempty = (len(value) > 0) if isinstance(value, (str, list)) else True
        
        if is_valid_type and is_nonempty:
            print(f"  {component}: OK")
        else:
            print(f"  {component}: INCOMPLETE")
            aiia_complete = False
    else:
        print(f"  {component}: MISSING")
        aiia_complete = False

# Check deployment recommendation
if 'deployment_recommendation' in aiia:
    rec = aiia['deployment_recommendation'].upper()
    if rec in ['DEPLOY', 'CONDITIONAL DEPLOY', 'DO NOT DEPLOY']:
        print(f"  deployment_recommendation: OK ({rec})")
    else:
        print(f"  deployment_recommendation: INVALID ({rec})")
        aiia_complete = False
else:
    print(f"  deployment_recommendation: MISSING")
    aiia_complete = False

if aiia_complete:
    print(f"\nRESULT: PASS — AIIA contains all five components with valid deployment recommendation.")
else:
    print(f"\nRESULT: FAIL — AIIA is incomplete.")

## Optional Challenges

**Challenge 1**: Recompute fairness metrics disaggregated by intersecting gender AND race (four groups: female + non-white, female + white, male + non-white, male + white). Do fairness disparities persist or change when you slice the data this way?

**Challenge 2**: Draft a model card for this classifier following the "Model Cards for Model Reporting" template (Mitchell et al., 2019, arXiv:1810.03993). Include sections on model type, training procedure, performance benchmarks disaggregated by demographic group, recommended use cases, and limitations.

**Challenge 3**: The AIIA states that fairness-constrained retraining is mitigation 1. Propose a specific constraint type (Demographic Parity, Equalized Odds, or Calibration) and explain why you chose it over the others for this hiring scenario.

## Completion Criteria

You have completed this lab successfully when:

1. **Step 1**: Accuracy values are computed overall and disaggregated by gender and race, with exact percentages reported.
2. **Step 2**: Demographic Parity and Equalized Odds differences are computed using fairness metrics, and SHAP feature importance is examined for transparency concerns.
3. **Step 3**: Audit report is completed with findings for all four dimensions (accuracy, fairness, transparency, accountability), each with an assigned status (pass/flag/fail). The verification cell prints PASS.
4. **Step 4**: AI Impact Assessment is drafted with all five components (system description, risk identification, affected populations, mitigation plan, residual risk statement) and concludes with a deployment recommendation (deploy / conditional deploy / do not deploy). The verification cell prints PASS.
5. **Comprehension Questions**: You have answered both comprehension questions in writing.
6. **Both verification cells run without error and print PASS** for audit report and AIIA completeness.